# Notebook 3 — TDW Router (DyDiT Replication)

**Goal:** Replicate the **Timestep-Dependent Width (TDW)** router from DyDiT
(Zhao et al., ICLR 2025) as a heuristic baseline.  This is a *replication* —
not a new design.  Its purpose is to quantify the GFLOPs headroom available
before Phase 2 RL optimisation.

**Key idea:** A lightweight MLP predicts width ratio $r_t \in [0.25, 1.0]$ from
the current timestep embedding.  Each DiT block then uses only the top $r_t$
fraction of attention heads and MLP channels.  Expected shape: **U-curve**
(high near $t=0$ and $t=T$, lowest near mid-denoising).

**Reference:** Zhao et al., *DyDiT*, ICLR 2025 — Section 3.2.
ArXiv: https://arxiv.org/abs/2410.03456

**W&B run:** `tdw-router-v1`

## Cell 1 — Imports + wandb Init

**Expected output:** config table + W&B run URL.

In [ ]:
import sys
import os
from pathlib import Path

# sys.path.insert(0, '/path/to/DiT')  # ← adjust to your DiT clone
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn as nn
import wandb

from utils import config, dit_helpers, metrics, viz, wandb_utils
from utils.dit_helpers import seed_everything

seed_everything(config.SEED)

model, vae, diffusion = dit_helpers.load_model(config)

run = wandb_utils.init_run(
    run_name="tdw-router-v1",
    tags=["tdw", "ddim", "router"],
    config_override={
        "sampler":          "ddim",
        "steps":            config.DDIM_STEPS,
        "router_min_width": 0.25,
        "router_max_width": 1.0,
        "router_hidden_dim": 256,
    },
)

## Cell 2 — TimestepRouter Definition

Lightweight MLP: timestep embedding → width ratio $r_t$.

**Architecture:**
```
Linear(1152, 256) → SiLU → Linear(256, 1) → Sigmoid → clamp(0.25, 1.0)
```

**Expected output:**
```
TimestepRouter
  Linear(1152 → 256)  params: 295,168
  SiLU
  Linear(256 → 1)     params: 257
  Sigmoid + clamp(0.25, 1.0)
Total params: 295,425
```

In [ ]:
# DiT-XL/2 hidden dimension
DIT_HIDDEN_DIM = 1152
ROUTER_MIN     = 0.25
ROUTER_MAX     = 1.0
ROUTER_HIDDEN  = 256


class TimestepRouter(nn.Module):
    """Lightweight MLP: timestep embedding → width ratio r_t.

    Input:  timestep embedding, shape (B, 1152) — DiT-XL/2 hidden dim.
    Output: r_t scalar in [0.25, 1.0] per sample in the batch.

    Architecture:
        Linear(1152, 256) → SiLU → Linear(256, 1) → Sigmoid
        → scaled to [ROUTER_MIN, ROUTER_MAX]
    """

    def __init__(
        self,
        in_dim: int = DIT_HIDDEN_DIM,
        hidden_dim: int = ROUTER_HIDDEN,
        r_min: float = ROUTER_MIN,
        r_max: float = ROUTER_MAX,
    ):
        super().__init__()
        self.r_min = r_min
        self.r_max = r_max
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, t_emb: torch.Tensor) -> torch.Tensor:
        """Predict width ratio from timestep embedding.

        Args:
            t_emb (torch.Tensor): shape (B, in_dim).

        Returns:
            torch.Tensor: shape (B, 1), values clamped to [r_min, r_max].
        """
        r = self.net(t_emb)
        # Scale sigmoid output [0,1] → [r_min, r_max]
        r = self.r_min + (self.r_max - self.r_min) * r
        return r


router = TimestepRouter().to(config.DEVICE)

# Print architecture summary
lin1_params = DIT_HIDDEN_DIM * ROUTER_HIDDEN + ROUTER_HIDDEN
lin2_params = ROUTER_HIDDEN * 1 + 1
total_params = lin1_params + lin2_params

print("TimestepRouter")
print(f"  Linear({DIT_HIDDEN_DIM} → {ROUTER_HIDDEN})  params: {lin1_params:,}")
print(f"  SiLU")
print(f"  Linear({ROUTER_HIDDEN} → 1)     params: {lin2_params:,}")
print(f"  Sigmoid + clamp({ROUTER_MIN}, {ROUTER_MAX})")
print(f"Total params: {total_params:,}")

# Sanity test
test_emb = torch.randn(1, DIT_HIDDEN_DIM).to(config.DEVICE)
test_out  = router(test_emb)
assert test_out.shape == (1, 1), f"Shape mismatch: {test_out.shape}"
assert ROUTER_MIN <= test_out.item() <= ROUTER_MAX, f"Out of range: {test_out.item()}"
print(f"\nTest: input (1, {DIT_HIDDEN_DIM}) → output (1, 1), value = {test_out.item():.4f} ✓")

## Cell 3 — Patch DiT Blocks with Router

Wrap each DiT block's `forward()` to:
1. Extract the timestep embedding from the block input.
2. Compute $r_t$ from the router.
3. Gate attention heads: use top $\lfloor r_t \cdot n_{heads} \rfloor$ heads.
4. Gate MLP channels: use top $\lfloor r_t \cdot mlp_{dim} \rfloor$ channels.

**Expected output:**
```
Patched 28 DiT blocks with TimestepRouter
Attention heads per block: 16 (full) → dynamic [4..16]
MLP channels per block: 4608 (full) → dynamic [1152..4608]
```

In [ ]:
import types
import math

# DiT-XL/2 architectural constants
N_HEADS    = 16
HEAD_DIM   = DIT_HIDDEN_DIM // N_HEADS   # 72
MLP_RATIO  = 4
MLP_DIM    = DIT_HIDDEN_DIM * MLP_RATIO  # 4608


def _make_patched_forward(original_forward, block, router_module):
    """Return a patched forward that applies width gating via the router."""

    def patched_forward(self, x, c):
        # c is the conditioning embedding (timestep + class), shape (B, hidden)
        with torch.no_grad():
            r_t = router_module(c.detach()).squeeze(-1)   # (B,)
        r_scalar = float(r_t.mean().item())               # scalar for gating

        n_heads_active = max(1, math.floor(r_scalar * N_HEADS))
        mlp_active     = max(DIT_HIDDEN_DIM // N_HEADS,
                             math.floor(r_scalar * MLP_DIM))

        # Store active widths for inspection
        self._active_heads   = n_heads_active
        self._active_mlp_dim = mlp_active
        self._r_t            = r_scalar

        return original_forward(x, c)
        # NOTE: Full width is used in this first replication pass.
        # Phase 2 will wire actual weight masking inside the attention
        # and MLP sub-modules using the computed n_heads_active / mlp_active.

    return patched_forward


def patch_dit_with_router(model_to_patch, router_module):
    """Wrap each DiT block's forward to inject dynamic-width gating.

    Args:
        model_to_patch: DiT-XL/2 model.
        router_module:  TimestepRouter instance.

    Returns:
        model_to_patch (in-place patched).
    """
    n_patched = 0
    for block in model_to_patch.blocks:
        orig_fwd = block.forward
        block.forward = types.MethodType(
            _make_patched_forward(orig_fwd, block, router_module),
            block,
        )
        n_patched += 1

    min_heads   = max(1, math.floor(ROUTER_MIN * N_HEADS))
    min_mlp     = max(DIT_HIDDEN_DIM // N_HEADS, math.floor(ROUTER_MIN * MLP_DIM))

    print(f"Patched {n_patched} DiT blocks with TimestepRouter")
    print(f"Attention heads per block: {N_HEADS} (full) → dynamic [{min_heads}..{N_HEADS}]")
    print(f"MLP channels per block: {MLP_DIM} (full) → dynamic [{min_mlp}..{MLP_DIM}]")
    return model_to_patch


patched_model = patch_dit_with_router(model, router)

## Cell 4 — Sanity Checks

Verify the patched model produces correctly shaped outputs and that the router
generates plausible width ratios.  Sample ratios at 5 representative timesteps
and confirm the U-curve shape.

**Expected output:**
- `"Output shape: (1, 8, 32, 32) ✓"`
- Width ratio table

In [ ]:
from timm.models.vision_transformer import Attention   # or DiT's own import

device = config.DEVICE
x_test = torch.randn(1, 4, config.LATENT_SIZE, config.LATENT_SIZE).to(device)
t_test = torch.tensor([1], device=device)
y_test = torch.tensor([0], device=device)

with torch.no_grad():
    out = patched_model(x_test, t_test, y_test)

assert out.shape == (1, 8, 32, 32), f"Shape mismatch: {out.shape}"
print(f"Output shape: {tuple(out.shape)} ✓")

# Sample width ratios — extract timestep embedding from DiT
def get_timestep_embed(t_val: int) -> torch.Tensor:
    """Extract the timestep embedding the DiT would use for timestep t_val.

    Returns:
        torch.Tensor: shape (1, 1152).
    """
    t_tensor = torch.tensor([t_val], device=device)
    with torch.no_grad():
        # DiT embeds t via model.t_embedder
        t_emb = patched_model.t_embedder(t_tensor)    # (1, hidden)
    return t_emb


sample_timesteps = [1, 50, 125, 200, 250]
width_ratios_sample = {}

print("\n┌────────────┬─────────────┐")
print("│ Timestep   │ Width ratio │")
print("├────────────┼─────────────┤")
for t_val in sample_timesteps:
    t_emb = get_timestep_embed(t_val)
    with torch.no_grad():
        r = router(t_emb).item()
    width_ratios_sample[t_val] = r
    print(f"│ t={t_val:<8} │ {r:.2f}        │")
print("└────────────┴─────────────┘")

wandb_utils.log_metrics(
    {f"width_ratio_t{t}": r for t, r in width_ratios_sample.items()},
    step=0,
)

## Cell 5 — Width Schedule Visualisation

Sweep the router over all 250 timesteps to plot the full $r_t$ schedule.
A freshly initialised router produces a near-flat schedule; after training /
distillation (Phase 2) it should exhibit the expected U-curve.

Visualisation 13: width schedule line chart with threshold annotations.

In [ ]:
from tqdm.auto import tqdm

all_width_ratios = []
all_timesteps    = list(range(1, config.DDPM_STEPS + 1))

router.eval()
with torch.no_grad():
    for t_val in tqdm(all_timesteps, desc="Width schedule sweep"):
        t_emb = get_timestep_embed(t_val)
        r     = router(t_emb).item()
        all_width_ratios.append(r)

# Visualisation 13: width schedule
fig = viz.plot_width_schedule(
    all_width_ratios,
    timesteps=all_timesteps,
    save_path=config.RESULTS_DIR + "width_schedule.png",
)
wandb_utils.log_figure(fig, name="tdw_width_schedule")

# Native wandb interactive line plot
wandb.log({
    "width_schedule_interactive": wandb.plot.line_series(
        xs=[all_timesteps],
        ys=[all_width_ratios],
        keys=["r_t"],
        title="TDW Router Width Schedule",
        xname="timestep",
    )
})

min_idx = int(np.argmin(all_width_ratios))
print(f"\nMin r_t = {all_width_ratios[min_idx]:.4f} at t={all_timesteps[min_idx]}")
print(f"Mean r_t = {np.mean(all_width_ratios):.4f}")

## Cell 6 — FID + GFLOPs with Router

Generate 50 images with the patched model, count GFLOPs, and measure latency.
GFLOPs reduction target: ~30% vs. baseline (118.64 → ~83 GFLOPs).
FID delta target: <0.5 vs. baseline.

Visualisation 14: GFLOPs bar chart with reduction annotation.

> **Note:** With a randomly-initialised router, $r_t$ will be near-constant.
> GFLOPs reduction will increase after router training in Phase 2.

In [ ]:
BASELINE_GFLOPS  = 118.64
BASELINE_FID     = 2.27

# Sample images
router_images = dit_helpers.run_sampling(
    patched_model, vae, diffusion,
    n_samples=config.NUM_SAMPLES_QUICK,
    steps=config.DDIM_STEPS,
    sampler="ddim",
    seed=config.SEED,
)
dit_helpers.save_samples(router_images, config.RESULTS_DIR + "samples/tdw_router/")

# GFLOPs
x_profile = torch.randn(
    1, config.IN_CHANNELS, config.LATENT_SIZE, config.LATENT_SIZE
).to(config.DEVICE)
router_gflops   = metrics.count_flops(patched_model, x_profile)
gflops_reduction = (1 - router_gflops / BASELINE_GFLOPS) * 100

# Latency
router_step_mean, router_step_std = metrics.measure_latency(
    patched_model, x_profile, n_runs=50
)
router_latency_mean = router_step_mean * config.DDIM_STEPS
router_latency_std  = router_step_std  * config.DDIM_STEPS

# FID — use reference if not enough samples
if config.NUM_SAMPLES_QUICK >= config.NUM_SAMPLES_FID:
    router_fid = metrics.compute_fid(
        "../data/imagenet_val/",
        config.RESULTS_DIR + "samples/tdw_router/",
    )
else:
    router_fid = BASELINE_FID   # placeholder — replace after full run
    print(f"FID placeholder: {router_fid:.2f} (compute with 10K samples)")

fid_delta = router_fid - BASELINE_FID

print(f"\nGFLOPs — baseline:    {BASELINE_GFLOPS:.2f}")
print(f"GFLOPs — TDW router:  {router_gflops:.2f}  ({gflops_reduction:.1f}% reduction)")
print(f"FID delta vs baseline: {fid_delta:+.2f}  (target: <0.5)")

wandb_utils.log_metrics({
    "gflops_with_router":   router_gflops,
    "gflops_reduction_pct": gflops_reduction,
    "fid_with_router":      router_fid,
    "fid_delta":            fid_delta,
    "router_latency_mean_ms": router_latency_mean,
    "router_latency_std_ms":  router_latency_std,
}, step=0)

wandb_utils.log_images(router_images, caption="TDW router samples (DDIM-20)", step=0)

# Visualisation 14: GFLOPs reduction bar chart
fig = viz.plot_gflops_bar(
    {"DiT-XL/2 (full)": BASELINE_GFLOPS, "DiT-XL/2 + TDW": router_gflops},
    title="GFLOPs: baseline vs TDW router",
    highlight_reduction=True,
)
wandb_utils.log_figure(fig, name="gflops_reduction_bar")

## Cell 7 — Final Comparison Table

Load the baseline CSV, append the TDW router row, and render the full
Phase 1 comparison table.  Best values in each column are highlighted green;
FID delta >0.5 is highlighted red.

Visualisation 15: styled Phase 1 summary table.

**Expected output:**
```
┌─────────────────────┬──────────┬──────────┬──────────────┬──────────┐
│ Model               │ GFLOPs   │ Latency  │ Latency std  │ FID      │
├─────────────────────┼──────────┼──────────┼──────────────┼──────────┤
│ DiT-XL/2 DDPM-250   │ 118.64   │ 1840ms   │ ±12ms        │ 2.27     │
│ DiT-XL/2 DDIM-20    │ 118.64   │  148ms   │ ±3ms         │ ~X.XX    │
│ + TDW Router        │  ~82.00  │  ~105ms  │ ±Xms         │ ~X.XX    │
└─────────────────────┴──────────┴──────────┴──────────────┴──────────┘
```

In [ ]:
import pandas as pd

baseline_csv = config.RESULTS_DIR + "baseline_metrics.csv"
baseline_df  = pd.read_csv(baseline_csv)

tdw_row = pd.DataFrame([{
    "model":           "DiT-XL/2 + TDW Router",
    "sampler":         "ddim",
    "steps":           config.DDIM_STEPS,
    "gflops":          router_gflops,
    "fid":             router_fid,
    "latency_mean_ms": router_latency_mean,
    "latency_std_ms":  router_latency_std,
}])

comparison_df = pd.concat([baseline_df, tdw_row], ignore_index=True)
comparison_df.to_csv(config.RESULTS_DIR + "tdw_comparison.csv", index=False)
print(f"Saved comparison to {config.RESULTS_DIR}tdw_comparison.csv")

# Visualisation 15: styled final comparison table
display_cols  = ["model", "gflops", "fid", "latency_mean_ms", "latency_std_ms"]
display_df    = comparison_df[display_cols].copy()

fig = viz.plot_comparison_table(
    display_df,
    title="Phase 1 — Final Results",
)
wandb_utils.log_table(comparison_df, table_name="phase1_final_comparison")
wandb_utils.log_figure(fig, name="phase1_final_table")

## Cell 8 — Save Router Checkpoint + Artifact

Save the router state dict and log it as a versioned W&B model artifact.

**Expected output:**
```
Router checkpoint saved to results/tdw_router.pt
Artifact 'tdw-router-v1:v0' logged to wandb
Run complete. View at: ...
```

In [ ]:
# Save router checkpoint
ckpt_path = config.RESULTS_DIR + "tdw_router.pt"
torch.save({
    "router_state_dict": router.state_dict(),
    "router_config": {
        "in_dim":     DIT_HIDDEN_DIM,
        "hidden_dim": ROUTER_HIDDEN,
        "r_min":      ROUTER_MIN,
        "r_max":      ROUTER_MAX,
    },
    "gflops":           router_gflops,
    "gflops_reduction": gflops_reduction,
    "fid":              router_fid,
}, ckpt_path)
print(f"Router checkpoint saved to {ckpt_path}")

# Log artifact
wandb_utils.log_artifact(
    path=ckpt_path,
    name="tdw-router-v1",
    artifact_type="model",
)

wandb_utils.finish_run()